# SysCraft MQTT Message Subscriber

Run this notebook in JupyterLab alongside the publisher to prove that the broker is delivering packets to `syscraft/arena/#`.

In [ ]:
# Run once if needed:
# %pip install paho-mqtt

In [ ]:
import getpass
import json
import os
import threading
import uuid

import paho.mqtt.client as mqtt

MQTT_HOST = os.getenv("SYSCRAFT_MQTT_HOST", "mqtt.conceptio.cloud")
MQTT_PORT = int(os.getenv("SYSCRAFT_MQTT_PORT", "1883"))
MQTT_USERNAME = os.getenv("SYSCRAFT_MQTT_USERNAME", "chris")
MQTT_PASSWORD = os.getenv("SYSCRAFT_MQTT_PASSWORD")
MQTT_CLIENT_ID = os.getenv("SYSCRAFT_MQTT_SUBSCRIBER_CLIENT_ID", f"syscraft-mqtt-observer-{uuid.uuid4().hex[:8]}")
MQTT_ROOT_TOPIC = os.getenv("SYSCRAFT_MQTT_ROOT_TOPIC", "syscraft/arena")
SUBSCRIPTION = f"{MQTT_ROOT_TOPIC.strip('/')}/#"

print(f"Broker: {MQTT_HOST}:{MQTT_PORT}")
print(f"Client ID: {MQTT_CLIENT_ID}")
print(f"Subscription: {SUBSCRIPTION}")

In [ ]:
subscriber = {"client": None, "connected": threading.Event(), "messages": 0}

def _pretty_payload(payload):
    text = payload.decode("utf-8", errors="replace")
    try:
        return json.dumps(json.loads(text), indent=2, sort_keys=True)
    except json.JSONDecodeError:
        return text

def start_subscriber():
    if subscriber["client"] is not None:
        print("Subscriber is already running.")
        return

    password = MQTT_PASSWORD or getpass.getpass("MQTT password: ")
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=MQTT_CLIENT_ID, protocol=mqtt.MQTTv5)
    client.username_pw_set(MQTT_USERNAME, password)

    def on_connect(mqtt_client, _userdata, _flags, reason_code, _properties):
        if reason_code.is_failure:
            print(f"Broker rejected connection: {reason_code}")
            return
        result, _message_id = mqtt_client.subscribe(SUBSCRIPTION, qos=1)
        if result != mqtt.MQTT_ERR_SUCCESS:
            print(f"Subscription request failed: rc={result}")
            return
        subscriber["connected"].set()
        print(f"Connected; subscription requested: {SUBSCRIPTION}")

    def on_subscribe(_client, _userdata, _message_id, reason_codes, _properties):
        if any(code.value >= 128 for code in reason_codes):
            print(f"Broker denied subscription: {reason_codes}")
        else:
            print(f"Subscribed: {SUBSCRIPTION}")

    def on_message(_client, _userdata, message):
        subscriber["messages"] += 1
        print(f"\n--- message {subscriber['messages']} ---")
        print(f"topic: {message.topic}")
        print(f"payload:\n{_pretty_payload(message.payload)}")

    def on_disconnect(_client, _userdata, _disconnect_flags, reason_code, _properties):
        print(f"Disconnected: {reason_code}")

    client.on_connect = on_connect
    client.on_subscribe = on_subscribe
    client.on_message = on_message
    client.on_disconnect = on_disconnect
    subscriber["connected"].clear()
    client.connect(MQTT_HOST, MQTT_PORT, 60)
    client.loop_start()
    subscriber["client"] = client
    print("Connecting...")

def stop_subscriber():
    client = subscriber["client"]
    if client is None:
        print("Subscriber is not running.")
        return
    client.loop_stop()
    client.disconnect()
    subscriber["client"] = None
    print(f"Stopped. Received {subscriber['messages']} message(s).")

## Start observing

Run this cell, then start the publisher notebook. The output will show each topic and JSON payload received from the broker.

In [ ]:
start_subscriber()

## Stop observing

In [ ]:
stop_subscriber()